# 🎬 TechnoCrazy — Generador de Videos Avatar (LivePortrait)
**Anima tu foto con audio → Video realista de tu cara hablando**

### Instrucciones:
1. Ejecuta cada celda en orden (Shift+Enter)
2. Sube tu foto y el audio cuando se pida
3. Descarga el video generado

> Runtime → Change runtime type → **T4 GPU** (gratis)

In [ ]:
# CELDA 1 — Verificar GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ Sin GPU — cambia el runtime a T4')

In [ ]:
# CELDA 2 — Instalar LivePortrait
import os

if not os.path.exists('LivePortrait'):
    !git clone https://github.com/KwaiVGI/LivePortrait.git
    %cd LivePortrait
    !pip install -q -r requirements.txt
    print('✅ LivePortrait instalado')
else:
    %cd LivePortrait
    print('✅ LivePortrait ya estaba instalado')

In [ ]:
# CELDA 3 — Descargar modelos (solo primera vez, ~2 min)
import os

if not os.path.exists('pretrained_weights'):
    !pip install -q huggingface_hub
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id='KwaiVGI/LivePortrait',
        local_dir='pretrained_weights',
        ignore_patterns=['*.git*']
    )
    print('✅ Modelos descargados')
else:
    print('✅ Modelos ya descargados')

In [ ]:
# CELDA 4 — Subir tu FOTO (imagen de Rafael)
from google.colab import files
from IPython.display import Image, display
import shutil, os

print('📸 Sube una foto tuya (JPG o PNG):')
print('   - Cara visible, iluminación buena')
print('   - Fondo no importa')
print('   - Usa cualquiera de tus fotos TechnoCrazy')

uploaded = files.upload()
foto_nombre = list(uploaded.keys())[0]
foto_path = f'/content/LivePortrait/assets/examples/source/{foto_nombre}'
os.makedirs(os.path.dirname(foto_path), exist_ok=True)
shutil.copy(foto_nombre, foto_path)

print(f'\n✅ Foto cargada: {foto_nombre}')
display(Image(foto_path, width=300))

In [ ]:
# CELDA 5 — Subir AUDIO (el script narrado por ElevenLabs o tu voz)
from google.colab import files
import shutil, os

print('🎙️ Sube el audio MP3/WAV con el script:')
print('   - Puede ser de ElevenLabs (voz clonada)')
print('   - O tu propia voz grabada')
print('   - Máximo recomendado: 60 segundos')

uploaded_audio = files.upload()
audio_nombre = list(uploaded_audio.keys())[0]
audio_path = f'/content/LivePortrait/assets/examples/driving/{audio_nombre}'
os.makedirs(os.path.dirname(audio_path), exist_ok=True)
shutil.copy(audio_nombre, audio_path)

print(f'\n✅ Audio cargado: {audio_nombre}')

In [ ]:
# CELDA 6 — GENERAR EL VIDEO (el paso principal ~2-5 min)
import subprocess, os

output_dir = '/content/LivePortrait/results'
os.makedirs(output_dir, exist_ok=True)

print('🎬 Generando video... (2-5 minutos)')
print('No cierres esta pestaña')

cmd = [
    'python', 'inference.py',
    '-s', foto_path,
    '-d', audio_path,
    '--output-dir', output_dir,
    '--flag-lip-zero',          # Mejor sincronización labial
    '--flag-relative-motion',   # Movimiento natural
    '--flag-pasteback'          # Mantiene fondo original
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    videos = [f for f in os.listdir(output_dir) if f.endswith('.mp4')]
    if videos:
        video_final = os.path.join(output_dir, videos[-1])
        print(f'\n✅ VIDEO GENERADO: {video_final}')
    else:
        print('⚠️ Revisar output:', result.stdout[-500:])
else:
    print('❌ Error:', result.stderr[-500:])

In [ ]:
# CELDA 7 — PREVISUALIZAR Y DESCARGAR
from IPython.display import HTML
from google.colab import files
import base64

# Mostrar video en notebook
with open(video_final, 'rb') as f:
    video_b64 = base64.b64encode(f.read()).decode()

display(HTML(f'''
<video width="360" controls>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
'''))

print('\n⬇️ Descargando video...')
files.download(video_final)
print('✅ Listo. Guarda el video en Google Drive para que n8n lo tome.')

In [ ]:
# CELDA 8 — GUARDAR DIRECTO EN GOOGLE DRIVE (opcional)
from google.colab import drive
import shutil, os
from datetime import datetime

drive.mount('/content/drive')

carpeta_drive = '/content/drive/MyDrive/TechnoCrazy/Contenido/Videos'
os.makedirs(carpeta_drive, exist_ok=True)

fecha = datetime.now().strftime('%Y%m%d_%H%M')
destino = f'{carpeta_drive}/TC_{fecha}.mp4'
shutil.copy(video_final, destino)

print(f'✅ Video guardado en Drive: {destino}')
print('n8n lo tomará automáticamente desde ahí')